# Plate Compression and Validation

This notebook handles plate compression and validation for sequencing pipelines:
- Read sample accession files and Qiita metadata files
- Assign compression layout (96→384 well mapping)
- Add controls (blanks, katharoseq)
- Validate plate dataframe

This notebook is **generic** and can be used as the first stage of multiple pipelines
(shotgun metagenomics, metatranscriptomics, etc.).

You'll start out with **sample accession file(s)**, which link each sample to its appropriate matrix tube barcode. Then you'll read in the sample information from Qiita metadata file(s) for the project(s) the samples come from. 

Next, you'll define a compression layout to generate a 384-well dataframe of the sample well-locations and associated extraction plate metadata [`Project Name`, `Project Plate`, `Project Abbreviation`, `Plate elution volume`] The compression step will import 96-well plate layouts from **VisionMate plate reader output files**, and will assign a 384-well sample well-location based on the 384-well quadrant (`Plate Position`) each 96-well plate is occupying. After that, you'll provide the directories containing your control matrix tubes, and the notebook will compare the matrix tube barcodes in your 384-well plate with those stored in folders documenting control matrix tubes to automatically assign controls using the add_controls() function.  Finally, the plate dataframe built up by the notebook will be validated.

## Inputs
- `expt_name` - experiment name
- `studies_info` - info about the sample accession and qiita metadata files for each study on the input plates
- `compression_layout` - list of plate info (including plate map file) and positions for each input plate
- `blanks_dir` - path to blanks directory
- `katharoseq_dir` - optional path to katharoseq controls directory
- `file_name_base` - base path for output files (e.g., './output/QC/MyExperiment')

## Outputs
- `{file_name_base}_plate_df_compvalid.txt` - compressed, validated plate dataframe with added controls
- `{file_name_base}_expt_info.yml` - experiment metadata for downstream notebooks

In [ ]:
%reload_ext watermark
%matplotlib inline

import yaml
from metapool.metapool import validate_plate_df
from metapool.util import (
    join_dfs_from_files, extend_sample_accession_df,
    extend_compression_layout_info, QIITA_STUDY_ID_KEY, warn_if_fp_exists)
from metapool import (add_controls, compress_plates,
                      TUBECODE_KEY, SAMPLE_NAME_KEY)
from metapool.mp_strings import PM_SAMPLE_KEY, PM_WELL_KEY
from metapool.notebook_utils import (
    get_studies_attr_list, pick_expected_separator)
%watermark -i -v -iv -m -h -p metapool,sample_sheet,openpyxl -u

In [ ]:
! conda list

## Step 0: Provide inputs

In [ ]:
# ## INPUT
# expt_name = "RKL4982"

# ## INPUT
# # Base path for output files (e.g., './output/QC/MyExperiment')
# file_name_base = "./test_output/QC/ShotgunMetag"

In [ ]:
# ## INPUT
# # One dictionary per study included in the samples on this run.
# studies_info = [
#     # EVERY entry in the dictionary must be specifically updated
#     # *every* time this notebook is run--none of these have defaults!
#     {
#         'Project Name': 'Celeste_Adaptation_12986', # PROJECTNAME_QIITAID
#         'Project Abbreviation': 'ADAPT', # PROJECTNAME
#         'sample_accession_fp': './test_data/Plate_Maps/sa_file_1.tsv',
#         'qiita_metadata_fp': './test_data/Plate_Maps/12986_20230314-090655.txt',
#         'experiment_design_description': 'isolate sequencing',
#         'HumanFiltering': 'False',
#         'Email': 'r@gmail.com'
#     },
#     {
#         'Project Name': 'TestProjB_10001', # PROJECTNAME_QIITAID
#         'Project Abbreviation': 'TestProjB', # PROJECTNAME
#         'sample_accession_fp': './test_data/Plate_Maps/sa_file_2.tsv',
#         'qiita_metadata_fp': './test_data/Plate_Maps/10001_20240503-090339.txt',
#         'experiment_design_description': 'whole genome sequencing',
#         'HumanFiltering': 'True',
#         'Email': 'l@ucsd.edu'
#     },
#     {
#         'Project Name': 'Celeste_Marmoset_14577', # PROJECTNAME_QIITAID
#         'Project Abbreviation': 'MARMO', # PROJECTNAME
#         'sample_accession_fp': './test_data/Plate_Maps/sa_file_3.tsv',
#         'qiita_metadata_fp': './test_data/Plate_Maps/14577_20230711-082202.txt',
#         'experiment_design_description': 'whole genome sequencing',
#         'HumanFiltering': 'False',
#         'Email': 'c@ucsd.edu'
#     }
# ]

In [ ]:
# ## INPUT
# # Enter the input 96-well plate info for each unique source plate
# # in the layout.  If you want to compress 4 separate 96-well plates,
# # enter 4 separate dictionaries (one for each quadrant) here. If you are doing
# # replicates, enter dictionaries ONLY for the quadrants in which the original
# # source plates are located, not for the destination quadrants they will be
# # replicated into (that is handled later).
# compression_layout = [
#     {
#         # top left plate
#         'Plate Position': 1, # as int
#         'Plate map file': './test_data/Plate_Maps/2022_summer_Celeste_Adaptation_16_plate_map.tsv',
#         'Project Name': 'Celeste_Adaptation_12986', # PROJECTNAME_QIITAID
#         'Project Plate': 'Plate_16', # Plate_#
#         'Plate elution volume': 110
#     },
#     {
#         # top right plate
#         'Plate Position': 2, # as int
#         'Plate map file': './test_data/Plate_Maps/2022_summer_Celeste_Adaptation_17_plate_map.tsv',
#         'Project Name': 'Celeste_Adaptation_12986', # PROJECTNAME_QIITAID
#         'Project Plate': 'Plate_17', # Plate_#
#         'Plate elution volume': 110
#     },
#     {
#         # bottom left plate
#         'Plate Position': 3, # as int
#         'Plate map file': './test_data/Plate_Maps/2022_summer_Celeste_Adaptation_18_plate_map.tsv',
#         'Project Name': 'Celeste_Adaptation_12986', # PROJECTNAME_QIITAID
#         'Project Plate': 'Plate_18', # Plate_#
#         'Plate elution volume': 110
#     },
#     {
#         # bottom right plate
#         'Plate Position': 4, # as int
#         'Plate map file': './test_data/Plate_Maps/TestProjB_1000_plate_map.tsv',
#         'Project Name': 'TestProjB_10001', # PROJECTNAME_QIITAID
#         'Project Plate': 'Plate_1000',  # Plate_#
#         'Plate elution volume': 110
#     },
# ]

In [ ]:
# ## INPUT
# blanks_dir = './test_data/BLANKS'

# ## INPUT
# # ATTENTION: Does your plate include katharoseq controls?
# # If *yes*, replace the None below with the path to the directory they are in, such as
# # katharoseq_dir = './test_data/katharoseq'
# katharoseq_dir = None

## Step 1: Read in sample accession files

In [ ]:
# read in the sample accession files
sample_accession_fps = get_studies_attr_list(
    studies_info, 'sample_accession_fp')
sample_acc_sep, sa_sep_name = pick_expected_separator(sample_accession_fps)
print(f"Expected sample accession separator: {sa_sep_name}")

In [ ]:
sample_accession_df = join_dfs_from_files(
    sample_accession_fps, [SAMPLE_NAME_KEY, TUBECODE_KEY], sep=sample_acc_sep)
sample_accession_df.shape

In [ ]:
sample_accession_df.head()

## Step 2: Read in the sample info from Qiita

In [ ]:
# read in the qiita metadata files
qiita_metadata_fps = get_studies_attr_list(studies_info, 'qiita_metadata_fp')
qiita_metadata_sep, qm_sep_name = pick_expected_separator(qiita_metadata_fps)
print(f"Expected qiita metadata separator: {qm_sep_name}")

In [ ]:
metadata_df = join_dfs_from_files(
    qiita_metadata_fps, [SAMPLE_NAME_KEY, QIITA_STUDY_ID_KEY],
    opt_cols_to_extract=['tube_id'], unique_cols=[SAMPLE_NAME_KEY],
    sep=qiita_metadata_sep)
metadata_df.shape

In [ ]:
metadata_df.head()

Now use the metadata to link the study info into the sample accession dataframe:

In [ ]:
extended_sample_accession_df = extend_sample_accession_df(
    sample_accession_df, studies_info, metadata_df)
extended_sample_accession_df.head()

## Step 3: Assign the compression layout and add controls

In [ ]:
# copy study info into the compression layout dictionary (so that it doesn't
# have to be entered manually in both places)
extended_compression_layout = extend_compression_layout_info(
    compression_layout, studies_info)

In [ ]:
plate_df = compress_plates(extended_compression_layout,
                           extended_sample_accession_df, well_col=PM_WELL_KEY)
plate_df.head()

Check for samples with missing names; at this point, we expect all blanks and katharoseq controls WON'T have names.

In [ ]:
def check_nan_samples(a_plate_df, a_blanks_dir=None):
    num_remaining_nans = a_plate_df[a_plate_df[PM_SAMPLE_KEY].isna()].shape[0]
    print("Number of samples with missing names: %d" % num_remaining_nans)

    if num_remaining_nans > 0 and a_blanks_dir:
        err_msg = f"""
By now, all samples should have names, so **do not continue** before fixing this!

"Unofficial" blanks are the most likely issue.
Determine if the tube codes for the problem samples (shown below) are blanks.
If they are, add them to the missing_blanks.csv file in the {a_blanks_dir} directory.
Then re-run from 'Step 3: Assign the compression layout and add controls'."""
        print(err_msg)

In [ ]:
check_nan_samples(plate_df)

In [ ]:
plate_df = add_controls(plate_df, blanks_dir, katharoseq_dir)

After adding controls, check again for samples with missing names; at this point, we expect all blanks and katharoseq controls WILL have names, so if there are any remaining samples without names, stop processing and fix them!

In [ ]:
## DECISION -- stop if there are still samples without names
check_nan_samples(plate_df, a_blanks_dir=blanks_dir)
plate_df[plate_df[PM_SAMPLE_KEY].isna()]

## Step 4: Validate plate dataframe

In [ ]:
# note that this function does not *need* the extended sample accession df,
# but it is easier to use it just to keep things consistent
validate_plate_df(plate_df, metadata_df, extended_sample_accession_df,
                  blanks_dir, katharoseq_dir)

## Step 5: Write output files

In [ ]:
# Construct output file paths from file_name_base
plate_df_fp = f"{file_name_base}_plate_df_compvalid.txt"
expt_info_fp = f"{file_name_base}_expt_info.yml"

In [ ]:
# Write plate_df to file
warn_if_fp_exists(plate_df_fp)
plate_df.to_csv(plate_df_fp, sep='\t', index=False)
print(f"Wrote plate_df to: {plate_df_fp}")

In [ ]:
# Save the experiment and study info for downstream notebooks
warn_if_fp_exists(expt_info_fp)
expt_info = {
    "experiment_name": expt_name,
    "studies": studies_info
}
with open(expt_info_fp, 'w') as file:
    yaml.dump(expt_info, file, default_flow_style=False, width=1000)
print(f"Wrote expt_info to: {expt_info_fp}")

In [ ]:
plate_df.head()